In [13]:
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil
import cv2
import rasterio
from pathlib import Path
import glob
from tqdm import tqdm
from image_processor import ImageProcessor, ThreeBandInputPaths

In [14]:
INPUT_PATH_SMALL = Path("/home/samuel/test/MasterThesis/Orthomosaics/small")
INPUT_PATH_MID  = Path("/home/samuel/test/MasterThesis/Orthomosaics/mid")
INPUT_PATH_LARGE = Path("/home/samuel/test/MasterThesis/Orthomosaics/large")

INPUT_PATHS = [INPUT_PATH_SMALL, INPUT_PATH_MID, INPUT_PATH_LARGE]

In [15]:
def norm_index_image(image, min, max):
    norm_image = np.clip(image, min, max)
    norm_image = ((norm_image - min) / (max - min) * 255).astype(np.uint8)
    return norm_image

In [16]:
extension = ".png"

In [5]:
ndvi_min = -1.00
ndvi_max = 1.00

rvi_min = 0.00
rvi_max = 9.00

rvi_min = -9000.00
rvi_max = 9000.00


for INPUT_PATH in INPUT_PATHS:

    subfolders = [f.name for f in INPUT_PATH.iterdir() if f.is_dir()]
    
    for subfolder in subfolders:
  
        subfolder_path = INPUT_PATH / subfolder

        subsubfolders = [f.name for f in subfolder_path.iterdir() if f.is_dir()]

        for subsubfolder in subsubfolders:

            image_path = subfolder_path / subsubfolder / "processed_output" / "image_tiles_indeces"
            image_path_utf8 = subfolder_path / subsubfolder / "processed_output" / "image_tiles_indeces_utf8"

            index_image_path = image_path / 'NDVI'
            index_image_path_utf8 = image_path_utf8 / 'NDVI'

            os.makedirs(index_image_path_utf8, exist_ok=True)

            ndvi_exists = False

            if len(glob.glob(os.path.join(index_image_path_utf8, f"*{extension}"))) > 0:
                print(f"Files already exist in {index_image_path_utf8} with extension {extension}")
                ndvi_exists = True

            if not ndvi_exists:
                print(f"Processing NDVI images in {index_image_path} and saving to {index_image_path_utf8}...")

                for filepath in tqdm(glob.glob(os.path.join(index_image_path, f"*{extension}"))):
                    filename = os.path.basename(filepath)
                    if "xml" in filename:
                        continue
                    with rasterio.open(filepath) as src:
                        img = src.read(1)
                    
                    ndvi = img.astype(np.float32)
                    # Convert index image to UTF-8 encoded image
                    ndvi_clipped = np.clip(ndvi, ndvi_min, ndvi_max)
                    ndvi_norm = np.floor(((ndvi_clipped + 1) / 2) * 255).astype(np.uint8)
                    output_path = os.path.join(index_image_path_utf8, filename)
                    with rasterio.open(output_path, 'w', driver='GTiff', height=ndvi_norm.shape[0],
                                    width=ndvi_norm.shape[1], count=1, dtype=ndvi_norm.dtype) as dst:
                        dst.write(ndvi_norm, 1)

            index_image_path = image_path / 'NGRDI'
            index_image_path_utf8 = image_path_utf8 / 'NGRDI'

            os.makedirs(index_image_path_utf8, exist_ok=True)

            ngrdi_exists = False

            if len(glob.glob(os.path.join(index_image_path_utf8, f"*{extension}"))) > 0:
                print(f"Files already exist in {index_image_path_utf8} with extension {extension}")
                ngrdi_exists = True

            if not ngrdi_exists:
                print(f"Processing NGRDI images in {index_image_path} and saving to {index_image_path_utf8}...")

                for filepath in tqdm(glob.glob(os.path.join(index_image_path, f"*{extension}"))):
                    filename = os.path.basename(filepath)
                    with rasterio.open(filepath) as src:
                        img = src.read(1)
                    
                    ndvi = img.astype(np.float32)
                    ndvi_clipped = np.clip(ndvi, ndvi_min, ndvi_max)
                    ndvi_norm = np.floor(((ndvi_clipped + 1) / 2) * 255).astype(np.uint8)
                output_path = os.path.join(index_image_path_utf8, filename)
                with rasterio.open(output_path, 'w', driver='GTiff', height=ndvi_norm.shape[0],
                                width=ndvi_norm.shape[1], count=1, dtype=ndvi_norm.dtype) as dst:
                    dst.write(ndvi_norm, 1)


            index_image_path = image_path / 'RVI'
            index_image_path_utf8 = image_path_utf8 / 'RVI'

            os.makedirs(index_image_path_utf8, exist_ok=True)

            rvi_exists = False

            if len(glob.glob(os.path.join(index_image_path_utf8, f"*{extension}"))) > 0:
                print(f"Files already exist in {index_image_path_utf8} with extension {extension}")
                rvi_exists = True

            if not rvi_exists:
                print(f"Processing RVI images in {index_image_path} and saving to {index_image_path_utf8}...")

                for filepath in tqdm(glob.glob(os.path.join(index_image_path, f"*{extension}"))):
                    filename = os.path.basename(filepath)
                    with rasterio.open(filepath) as src:
                        img = src.read(1)
                    
                    rvi = img.astype(np.float32)
                    rvi_norm = norm_index_image(rvi, rvi_min, rvi_max)

                    output_path = os.path.join(index_image_path_utf8, filename)
                    with rasterio.open(output_path, 'w', driver='GTiff', height=rvi_norm.shape[0],
                                    width=rvi_norm.shape[1], count=1, dtype=rvi_norm.dtype) as dst:
                        dst.write(rvi_norm, 1)


            index_image_path = image_path / 'EXGR'
            index_image_path_utf8 = image_path_utf8 / 'EXGR'
            
            os.makedirs(index_image_path_utf8, exist_ok=True)

            exgr_exists = False

            if len(glob.glob(os.path.join(index_image_path_utf8, f"*{extension}"))) > 0:
                print(f"Files already exist in {index_image_path_utf8} with extension {extension}")
                exgr_exists = True

            if not exgr_exists:
                print(f"Processing EXGR images in {index_image_path} and saving to {index_image_path_utf8}...")

                for filepath in tqdm(glob.glob(os.path.join(index_image_path, f"*{extension}"))):
                    filename = os.path.basename(filepath)
                    with rasterio.open(filepath) as src:
                        img = src.read(1)
                    
                    rvi = img.astype(np.float32)
                    rvi_norm = norm_index_image(rvi, rvi_min, rvi_max)

                    output_path = os.path.join(index_image_path_utf8, filename)
                    with rasterio.open(output_path, 'w', driver='GTiff', height=rvi_norm.shape[0],
                                    width=rvi_norm.shape[1], count=1, dtype=rvi_norm.dtype) as dst:
                        dst.write(rvi_norm, 1)
    



Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/original/original/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/original/original/processed_output/image_tiles_indeces_utf8/NGRDI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/original/original/processed_output/image_tiles_indeces_utf8/RVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/original/original/processed_output/image_tiles_indeces_utf8/EXGR with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated_rotated/translated_500x_500y_rotated_45/processed_output/image_tiles_indeces_utf8/NDVI with extension .png
Files already exist in /home/samuel/test/MasterThesis/Orthomosaics/small/translated_rotated/translated_500x_500y_rotated_45/processed_output/image_tiles_indeces_utf8/NGRDI with 

In [73]:
# get all input data from the large field dataset
BASE_INPUT_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/")

sub = ["large", "mid", "small"]
label_paths = []
image_paths = []
output_paths = []

for s in sub:
    print(f"Processing {s} field dataset")
    BASE_INPUT_PATH = Path(f"/home/samuel/test/MasterThesis/Orthomosaics/{s}")

    folders = os.listdir(BASE_INPUT_PATH)
    input_paths = []
    for folder in folders:
        folder_path = BASE_INPUT_PATH / folder
        if folder_path.is_dir():
                input_paths.append(folder_path)

    # find sub_folders in the large field dataset
    sub_folders = []
    for folder in input_paths:
        for sub_folder in folder.iterdir():
            if sub_folder.is_dir():
                sub_folders.append(sub_folder)

    # find all label and image files in the sub_folders

    image_count = 0

    for sub_folder in sub_folders:
        output_paths.append(sub_folder / "combined_output")
        label_path = sub_folder / "labels"
        image_path = sub_folder / "processed_output"
        label_paths.append(label_path)
        image_paths.append(image_path)

        print("Checking image path:", image_path)

        image_path = image_path / "image_tiles"
        image_count += len(list(image_path.glob("*.tif")))

    print(f"Found {image_count} images")

print(len(label_paths), len(image_paths), len(output_paths))

PATHS = [image_paths, output_paths]

proc = ImageProcessor(BASE_INPUT_PATH, BASE_INPUT_PATH, "")


Processing large field dataset
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated_rotated/translated_500x_500y_rotated_45/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated_rotated/translated_250x_250y_rotated_45/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x250y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated500x250y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated250x500y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/translated/translated500x500y/processed_output
Checking image path: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/vertical/processed_output
Checki

In [8]:
# "RED_GREEN_NIR" combination (nir, red, green)
ending = "RED_GREEN_NIR"

input_paths = PATHS[0]
output_paths = PATHS[1]

recompile = False

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):
    print(f"Processing input directory: {input_dir} to output directory: {output_dir}")
    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_GREEN_NIR image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")

            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir /  "extended_green" / img_name_eg,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending,
                percentiles=[]
            )

    else:
        print("RED_GREEN_NIR image creation skipped; output directory already exists.")


# RED_NIR_NDVI combination (nir, red, ndvi)
ending = "RED_NIR_NDVI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_NIR_NDVI image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_ndvi = img_name.replace(".tif", "_ndvi.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "nir" / img_name_nir,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NDVI" / img_name_ndvi,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("RED_NIR_NDVI image creation skipped; output directory already exists.")

# RED_REDEDGE_NIR combination (Nir, red edge, red)
ending = "RED_REDEDGE_NIR"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_REDEDGE_NIR image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_re = img_name.replace(".tif", "_REDEDGE.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "red_edge" / img_name_re,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("RED_REDEDGE_NIR image creation skipped; output directory already exists.")

# DRG combination (NDVI, red, green)
ending = "RED_GREEN_NDVI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="RED_GREEN_NDVI image creation"):
            if "xml" in img_name:
                continue
            img_name_ndvi = img_name.replace(".tif", "_ndvi.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "extended_red" / img_name_er,
                    band2=input_dir / "extended_green" / img_name_eg,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NDVI" / img_name_ndvi
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("RED_GREEN_NDVI image creation skipped; output directory already exists.")

# DO nen with zeroed blue channel
ending = "NIR_RED_NO_BLUE"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NIR_RED_NO_BLUE image creation"):
            if "xml" in img_name:
                continue
            img_name_ndvi = img_name.replace(".tif", "_ngrdi.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "nir" / img_name_nir,
                    band2=input_dir / "extended_red" / img_name_er,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ndvi,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, True]
            )

    else:
        print("NIR_RED_NO_BLUE image creation skipped; output directory already exists.")

# NIR_RED_NGRDI combination (nir, red, ngrdi)
ending = "NIR_RED_NGRDI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NIR_RED_NGRDI image creation"):
            if "xml" in img_name:
                continue
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            img_name_er = img_name.replace(".tif", "_EXTEND_RED.png")
            img_name_ngrdi = img_name.replace(".tif", "_ngrdi.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "nir" / img_name_nir,
                    band2=input_dir / "extended_red" / img_name_er,
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ngrdi,
                ),
                output_path=dir,
                ending=ending
            )

    else:
        print("NIR_RED_NGRDI image creation skipped; output directory already exists.")

# DO nen with zeroed blue channel
ending = "NIR_GREEN_NO_BLUE"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NIR_GREEN_NO_BLUE image creation"):
            if "xml" in img_name:
                continue
            img_name_ndvi = img_name.replace(".tif", "_ngrdi.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ndvi,
                    band2=input_dir / "extended_green" / img_name_eg,
                    band1=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, True]
            )

    else:
        print("NIR_GREEN_NO_BLUE image creation skipped; output directory already exists.")

# DO nen with zeroed blue channel
ending = "NGRDI_NDVI_RVI"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)
    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="NGRDI_NDVI_RVI image creation"):
            if "xml" in img_name:
                continue
            img_name_ngrdi = img_name.replace(".tif", "_ngrdi.png")
            img_name_ndvi = img_name.replace(".tif", "_ndvi.png")
            img_name_rvi = img_name.replace(".tif", "_rvi.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band3=input_dir / "image_tiles_indeces_utf8" / "NGRDI" / img_name_ngrdi,
                    band2=input_dir / "image_tiles_indeces_utf8" / "NDVI"  / img_name_ndvi,
                    band1=input_dir / "image_tiles_indeces_utf8" / "RVI"  / img_name_rvi,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, False]
            )

    else:
        print("NGRDI_NDVI_RVI image creation skipped; output directory already exists.")

    # DO nen with zeroed blue channel
ending = "EXGR_GREEN_NIR"

input_paths = PATHS[0]
output_paths = PATHS[1]

for input_dir, output_dir in zip(sorted(input_paths), sorted(output_paths)):

    dir = Path(str(output_dir) + "/" + ending)

    if not dir.exists() or recompile:
        proc.set_input_path(input_dir / "image_tiles")

        for img_name in tqdm(sorted(os.listdir(proc.input_path)), desc="EXGR_GREEN_NIR image creation"):
            if "xml" in img_name:
                continue
            img_name_exgr = img_name.replace(".tif", "_exgr.png")
            img_name_eg = img_name.replace(".tif", "_EXTEND_GREEN.png")
            img_name_nir = img_name.replace(".tif", "_NIR.png")
            proc.calculate_three_band_image(
                input_paths = ThreeBandInputPaths(
                    band1=input_dir / "image_tiles_indeces_utf8" / "EXGR" / img_name_exgr,
                    band2=input_dir / "extended_green" / img_name_eg,
                    band3=input_dir / "nir" / img_name_nir,
                ),
                output_path=dir,
                ending=ending,
                do_zeros=[False, False, False]
            )

    else:
        print("EXGR_GREEN_NIR image creation skipped; output directory already exists.")

Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal_vertical/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/horizontal_vertical/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/vertical/processed_output to output directory: /home/samuel/test/MasterThesis/Orthomosaics/large/augmented/vertical/combined_output
RED_GREEN_NIR image creation skipped; output directory already exists.
Processing input directory: /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/proce

EXGR_GREEN_NIR image creation:   0%|          | 0/468 [00:00<?, ?it/s]

/home/samuel/test/MasterThesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
EXGR_GREEN_NIR image creation: 100%|██████████| 56/56 [00:03<00:00, 17.89it/s]


In [19]:
# rename file in combined_output/{COMBINATUION} based on whether they ar ein large, mid, small
# reanme to e.g. large_tile_x_y.png and large_tile_x_y.txt for labels

import re
import os
import glob
from pathlib import Path

COMBINATION = "NIR_RED_NGRDI"

LARGE_INPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/large")
MID_INPUT_FOLDER_PATH   = Path("/home/samuel/test/MasterThesis/Orthomosaics/mid")
SMALL_INPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/small")

BASE_INPUT_PATHS = [LARGE_INPUT_FOLDER_PATH,
                    MID_INPUT_FOLDER_PATH,
                    SMALL_INPUT_FOLDER_PATH]

# Regex to extract tile_x_y
tile_regex = re.compile(r"(tile[_-]?\d+[_-]\d+)", re.IGNORECASE)

for BASE_INPUT_PATH in BASE_INPUT_PATHS:

    last_folder_name = BASE_INPUT_PATH.name
    print(f"\nProcessing base folder: {last_folder_name}")

    for sub_folder in BASE_INPUT_PATH.iterdir():
        if not sub_folder.is_dir():
            continue

        for sub_sub_folder in sub_folder.iterdir():
            if not sub_sub_folder.is_dir():
                continue

            image_path = sub_sub_folder / "processed_output" / "rgb"
            label_path = sub_sub_folder / "labels_aabb"

            image_files = sorted(glob.glob(str(image_path / "*.png")))

            for image_file in image_files:

                image_filename = os.path.basename(image_file)

                # ── Extract tile_x_y using regex ─────────────────────
                match = tile_regex.search(image_filename)

                if not match:
                    print(f"Skipping (no tile found): {image_filename}")
                    continue

                tile_name = match.group(1)

                # normalize possible dashes
                tile_name = tile_name.replace("-", "_")

                new_image_filename = f"{last_folder_name}_{tile_name}.png"
                new_label_filename = f"{last_folder_name}_{tile_name}.txt"

                print(f"{image_filename}  →  {new_image_filename}")

                # rename image
                os.rename(
                    image_file,
                    image_path / new_image_filename
                )

                # rename corresponding label if exists
                old_label_path = label_path / f"{Path(image_filename).stem}.txt"
                new_label_path = label_path / new_label_filename

                if old_label_path.exists():
                    os.rename(old_label_path, new_label_path)



Processing base folder: large
Bjornkjaervej_TestFlight_2_bigger_tile_10_10_rgb.png  →  large_tile_10_10.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_11_rgb.png  →  large_tile_10_11.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_12_rgb.png  →  large_tile_10_12.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_13_rgb.png  →  large_tile_10_13.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_2_rgb.png  →  large_tile_10_2.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_3_rgb.png  →  large_tile_10_3.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_4_rgb.png  →  large_tile_10_4.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_5_rgb.png  →  large_tile_10_5.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_6_rgb.png  →  large_tile_10_6.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_7_rgb.png  →  large_tile_10_7.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_8_rgb.png  →  large_tile_10_8.png
Bjornkjaervej_TestFlight_2_bigger_tile_10_9_rgb.png  →  large_tile_10_9.png
Bjornkjaervej_TestFlight_2_bigger_tile_11_10_rgb.

In [20]:
import random
from pathlib import Path
import shutil

COMBINATION = "rgb"

OUTPUT_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/dataset_rgb_non_augmented")

LARGE_INPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/large")
MID_INPUT_FOLDER_PATH   = Path("/home/samuel/test/MasterThesis/Orthomosaics/mid")
SMALL_INPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/small")

BASE_INPUT_PATHS = [
    LARGE_INPUT_FOLDER_PATH,
    MID_INPUT_FOLDER_PATH,
    SMALL_INPUT_FOLDER_PATH
]

test_percentage = 0.20
val_percentage  = 0.20
random.seed(42)

# ─────────────────────────────────────────────
# STEP 1: Collect ORIGINAL stems only
# ─────────────────────────────────────────────

all_original_stems = []

for base in BASE_INPUT_PATHS:
    original_folder = base / "original" / "original"
    image_folder = original_folder / "processed_output" / COMBINATION

    for img in image_folder.glob("*.png"):
        all_original_stems.append(img.stem)

print(f"Total original stems: {len(all_original_stems)}")

# ─────────────────────────────────────────────
# STEP 2: Shuffle + Split
# ─────────────────────────────────────────────

random.shuffle(all_original_stems)

n_total = len(all_original_stems)
n_test  = int(n_total * test_percentage)
n_val   = int(n_total * val_percentage)

test_stems = set(all_original_stems[:n_test])
val_stems  = set(all_original_stems[n_test:n_test+n_val])
train_stems = set(all_original_stems[n_test+n_val:])

print(f"Train stems: {len(train_stems)}")
print(f"Val stems  : {len(val_stems)}")
print(f"Test stems : {len(test_stems)}")

# ─────────────────────────────────────────────
# STEP 3: Assign all data (with augmentation control)
# ─────────────────────────────────────────────

allow_train_augmentation = False
allow_validation_augmentation = False
allow_test_augmentation = False   # IMPORTANT

dataset = {
    "train": {"images": [], "labels": []},
    "val":   {"images": [], "labels": []},
    "test":  {"images": [], "labels": []},
}

for base in BASE_INPUT_PATHS:

    print(f"\nProcessing base folder: {base.name}")

    for subfolder in base.iterdir():
        if not subfolder.is_dir():
            continue

        for sub_sub_folder in subfolder.iterdir():
            if not sub_sub_folder.is_dir():
                continue

            is_augmented = sub_sub_folder.name != "original"

            image_folder = sub_sub_folder / "processed_output" / COMBINATION
            label_folder = sub_sub_folder / "labels_aabb"

            if not image_folder.exists():
                continue

            for img in image_folder.glob("*.png"):

                stem = img.stem
                label = label_folder / f"{stem}.txt"

                if not label.exists():
                    continue

                # TEST
                if stem in test_stems:
                    if is_augmented and not allow_test_augmentation:
                        continue
                    dataset["test"]["images"].append(img)
                    dataset["test"]["labels"].append(label)

                # VAL
                elif stem in val_stems:
                    if is_augmented and not allow_validation_augmentation:
                        continue
                    dataset["val"]["images"].append(img)
                    dataset["val"]["labels"].append(label)

                # TRAIN
                elif stem in train_stems:
                    if is_augmented and not allow_train_augmentation:
                        continue
                    dataset["train"]["images"].append(img)
                    dataset["train"]["labels"].append(label)

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────

print("\nFinal split summary:")
for split in dataset:
    print(
        f"{split.upper():5} → "
        f"{len(dataset[split]['images'])} images"
    )

print("\nFinal split label summary:")
# find the number of instances in each split
for split in dataset:
    total_instances = 0
    for label_path in dataset[split]["labels"]:
        with open(label_path, "r") as f:
            lines = f.readlines()
            total_instances += len(lines)
    print(f"{split.upper():5} → {total_instances} instances")

print(f"Test dataset example:  {dataset['test']['images'][0]} with label {dataset['test']['labels'][0]}")

print("\nLeakage check...")

train_set = set([p.stem for p in dataset["train"]["images"]])
val_set   = set([p.stem for p in dataset["val"]["images"]])
test_set  = set([p.stem for p in dataset["test"]["images"]])

print("Train ∩ Test:", len(train_set & test_set))
print("Val   ∩ Test:", len(val_set & test_set))
print("Train ∩ Val :", len(train_set & val_set))

for split in dataset:

    split_img_dir = OUTPUT_PATH / "images" / split
    split_label_dir = OUTPUT_PATH / "labels" / split

    split_img_dir.mkdir(parents=True, exist_ok=True)
    split_label_dir.mkdir(parents=True, exist_ok=True)

    for img_path, label_path in zip(dataset[split]["images"],
                                    dataset[split]["labels"]):

        stem = img_path.stem

        # get augmentation folder name
        # e.g. rotated45 / original / augmented
        augmentation_name = img_path.parent.parent.parent.name

        new_image_name = f"{stem}__{augmentation_name}.png"
        new_label_name = f"{stem}__{augmentation_name}.txt"

        shutil.copy(img_path, split_img_dir / new_image_name)
        shutil.copy(label_path, split_label_dir / new_label_name)

Total original stems: 859
Train stems: 517
Val stems  : 171
Test stems : 171

Processing base folder: large

Processing base folder: mid

Processing base folder: small

Final split summary:
TRAIN → 517 images
VAL   → 171 images
TEST  → 171 images

Final split label summary:
TRAIN → 972 instances
VAL   → 301 instances
TEST  → 295 instances
Test dataset example:  /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/processed_output/rgb/large_tile_1_11.png with label /home/samuel/test/MasterThesis/Orthomosaics/large/original/original/labels_aabb/large_tile_1_11.txt

Leakage check...
Train ∩ Test: 0
Val   ∩ Test: 0
Train ∩ Val : 0


In [ ]:
# create dataset split with training being the large augmented field, validation being the mid original field and test being the small original field

TRAINING_BASE_INPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/large")
VALIDATION_BASE_INPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/mid/original/original")
TEST_BASE_INPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/small/original/original")
OUTPUT_FOLDER_PATH = Path("/home/samuel/test/MasterThesis/Orthomosaics/dataset")

COMBINATION = "NIR_RED_NO_BLUE"

INPUT_FOLDER_PATHS = [TRAINING_BASE_INPUT_FOLDER_PATH, VALIDATION_BASE_INPUT_FOLDER_PATH, TEST_BASE_INPUT_FOLDER_PATH]
counter = 0

restart = True

if restart == True:

    for folder in tqdm(TRAINING_BASE_INPUT_FOLDER_PATH.iterdir(), f"Processing folders in {TRAINING_BASE_INPUT_FOLDER_PATH}"):
        folder_path = folder
        if folder.is_dir():
            subfolders = list(folder.iterdir())
            for subfolder in subfolders:
                subfolder_path = subfolder
                if os.path.isdir(subfolder_path):
                    image_files_path = os.path.join(subfolder_path, "combined_output", COMBINATION)
                    label_files_path = os.path.join(subfolder_path, "labels_new")
                    image_files = glob.glob(os.path.join(image_files_path, "*.png"))
                    label_files = glob.glob(os.path.join(label_files_path, "*.txt"))

                    image_files = sorted(image_files)
                    label_files = sorted(label_files)

                    # find matching image and label files
                    for label_file in label_files:
                        label_filename = os.path.basename(label_file)[:-4]  # remove .txt extension
                        for image_file in image_files:
                            if label_filename in image_file:
                                # change image file name to match label file name + add counter to the end of the file name
                                new_image_file_name = f"{label_filename}_{counter}.png"
                                counter += 1
                                new_image_file_path = os.path.join(OUTPUT_FOLDER_PATH, "images/train", new_image_file_name)
                                new_label_file_path = os.path.join(OUTPUT_FOLDER_PATH, "labels/train", new_image_file_name[:-4] + ".txt")
                                os.makedirs(os.path.dirname(new_image_file_path), exist_ok=True)
                                os.makedirs(os.path.dirname(new_label_file_path), exist_ok=True)
                                shutil.copy(image_file, new_image_file_path)
                                shutil.copy(label_file, new_label_file_path)
                                break

folder = VALIDATION_BASE_INPUT_FOLDER_PATH
label_files_path = os.path.join(folder, "labels_new")
image_files_path = os.path.join(folder, "combined_output", COMBINATION)

print(f"Processing folder: {folder}")
print(f"Image files path: {image_files_path}")
print(f"Label files path: {label_files_path}")

image_files = glob.glob(os.path.join(image_files_path, "*.png"))
label_files = glob.glob(os.path.join(label_files_path, "*.txt"))

image_files = sorted(image_files)
label_files = sorted(label_files)

for label_file in label_files:
    label_filename = os.path.basename(label_file)[:-4]  # remove .txt extension
    for image_file in image_files:
        if label_filename in image_file:
            # change image file name to match label file name + add counter to the end of the file name
            new_image_file_name = f"{label_filename}_{counter}.png"
            counter += 1
            new_image_file_path = os.path.join(OUTPUT_FOLDER_PATH, "images/val", new_image_file_name)
            new_label_file_path = os.path.join(OUTPUT_FOLDER_PATH, "labels/val", new_image_file_name[:-4] + ".txt")
            os.makedirs(os.path.dirname(new_image_file_path), exist_ok=True)
            os.makedirs(os.path.dirname(new_label_file_path), exist_ok=True)
            shutil.copy(image_file, new_image_file_path)
            shutil.copy(label_file, new_label_file_path)
            break


folder = TEST_BASE_INPUT_FOLDER_PATH
label_files_path = os.path.join(folder, "labels_new")
image_files_path = os.path.join(folder, "combined_output", COMBINATION)
image_files = glob.glob(os.path.join(image_files_path, "*.png"))
label_files = glob.glob(os.path.join(label_files_path, "*.txt"))

image_files = sorted(image_files)
label_files = sorted(label_files)


for label_file in label_files:
    label_filename = os.path.basename(label_file)[:-4]  # remove .txt extension
    for image_file in image_files:
        if label_filename in image_file:
            # change image file name to match label file name + add counter to the end of the file name
            new_image_file_name = f"{label_filename}_{counter}.png"
            counter += 1
            new_image_file_path = os.path.join(OUTPUT_FOLDER_PATH, "images/test", new_image_file_name)
            new_label_file_path = os.path.join(OUTPUT_FOLDER_PATH, "labels/test", new_image_file_name[:-4] + ".txt")
            os.makedirs(os.path.dirname(new_image_file_path), exist_ok=True)
            os.makedirs(os.path.dirname(new_label_file_path), exist_ok=True)
            shutil.copy(image_file, new_image_file_path)
            shutil.copy(label_file, new_label_file_path)
            break


Processing folders in /home/samuel/test/MasterThesis/Orthomosaics/large: 5it [00:00, 23.92it/s]


Processing folder: /home/samuel/test/MasterThesis/Orthomosaics/mid/original/original
Image files path: /home/samuel/test/MasterThesis/Orthomosaics/mid/original/original/combined_output/NIR_RED_NGRDI
Label files path: /home/samuel/test/MasterThesis/Orthomosaics/mid/original/original/labels_new


In [11]:
def create_data_yaml(output_path: Path, classes):
    """
    Creates a dataset.yaml file for the sorted dataset.
    """

    dataset_yaml = output_path / "dataset.yaml"
    with open(dataset_yaml, "w") as f:
        f.write(
f"""
path: {output_path}
train: images/train
val: images/val
test: images/test
nc: 1
names: {classes}
"""
)
    print(f"Created dataset.yaml at {dataset_yaml}")
